# Lista 2, część II: analiza własności modeli językowych

Modele (polskojęzyczne odpowiedniki modeli z artykułów):
- **BERT**: `allegro/herbert-base-cased` (12 warstw, d = 768, 12 głów, słownik BPE 50 tys.),
- **GPT-2**: `sdadas/polish-gpt2-medium` (24 warstwy, d = 1024, 16 głów, słownik BPE 51,2 tys.).

Dane: `clarin-knext/wsd_polish_datasets`, korpus **Składnica** (11,5 tys. zdań z NKJP, ręcznie oznaczone sensy z plWordNetu).

1. Odtworzenie badania Ethayarajha (2019), *How Contextual are Contextualized Word Representations?*: anizotropia, self-similarity, intra-sentence similarity, MEV, a dodatkowo (dzięki anotacji sensów) podobieństwo wystąpień tego samego i różnych sensów.
2. Parameter Projection (Dar i in. 2022, *Analyzing Transformers in Embedding Space*): listy par tokenów z macierzy $W_{VO}$ (załącznik C.1) i $W_{QK}$ (załącznik C.2).

## Teoria

### Ethayarajh (2019): jak bardzo kontekstowe są reprezentacje kontekstowe?

Niech słowo $w$ występuje w zdaniach $s_1, \dots, s_n$ na pozycjach $i_1, \dots, i_n$, a $f_\ell(s, i)$ to reprezentacja $s[i]$ w warstwie $\ell$.

- **Anizotropia (baseline)**: średni cosinus między reprezentacjami losowych wystąpień losowych słów (z różnych kontekstów). Dla przestrzeni izotropowej $\approx 0$, im bliżej 1, tym węższy „stożek”, w którym leżą wszystkie wektory.
- **Self-similarity**: $\mathrm{SelfSim}_\ell(w) = \frac{1}{n^2 - n} \sum_j \sum_{k \ne j} \cos(f_\ell(s_j, i_j), f_\ell(s_k, i_k))$, czyli średni cosinus między wystąpieniami tego samego słowa w różnych kontekstach. 1 = brak kontekstowości (wektor statyczny).
- **Intra-sentence similarity**: $\mathrm{IntraSim}_\ell(s) = \frac{1}{n} \sum_i \cos(\vec{s}_\ell, f_\ell(s, i))$, gdzie $\vec{s}_\ell$ to średnia wektorów słów zdania. Mówi, *jak* przejawia się kontekstowość: czy słowa jednego zdania zbiegają się do wspólnego punktu.
- **Maximum explainable variance**: $\mathrm{MEV}_\ell(w) = \sigma_1^2 / \sum_i \sigma_i^2$ dla macierzy wystąpień $w$, czyli górne ograniczenie tego, jak dobrze jeden statyczny wektor może zastąpić reprezentacje kontekstowe.
- Każdą miarę **koryguje się o anizotropię**: odejmuje się jej wartość dla losowych słów w tej samej warstwie (np. $\mathrm{SelfSim}^*_\ell(w) = \mathrm{SelfSim}_\ell(w) - \text{baseline}_\ell$).

Wnioski z artykułu (ELMo, BERT-base, GPT-2 small, zdania z SemEval STS, słowa w ≥ 5 kontekstach):
1. Reprezentacje są **anizotropowe we wszystkich warstwach poza wejściową**, i to tym bardziej, im wyższa warstwa. W ostatniej warstwie GPT-2 dwa losowe słowa mają cosinus prawie 1.
2. **Wyższe warstwy są bardziej kontekstowe** (niższe self-similarity). GPT-2 jest najbardziej kontekstowy.
3. **Najbardziej kontekstowe są słowa funkcyjne** (*the, of, to*), choć nie są wieloznaczne. O zmienności decyduje różnorodność kontekstów, a nie liczba sensów. Modele nie wybierają więc jednego z kilku „wektorów-sensów”.
4. Kontekstowość przejawia się różnie: w ELMo słowa jednego zdania upodabniają się do siebie, w BERT oddalają się (ale wciąż są bliżej niż losowe słowa), w GPT-2 są tak samo odległe jak losowe słowa.
5. Po korekcie średnio **< 5% wariancji** reprezentacji słowa wyjaśnia pierwsza składowa główna, więc statyczny wektor jest słabym zamiennikiem. Mimo to statyczne wektory z PC niskich warstw BERT-a są lepsze od GloVe i fastText na benchmarkach podobieństwa.

### Dar i in. (2022): Transformery w przestrzeni osadzeń (Parameter Projection)

Punkt wyjścia: stan ukryty $h$ można rzutować na słownik przez macierz osadzeń $E \in \mathbb{R}^{|V| \times d}$ (*logit lens*, $hE^T$). Skoro wszystkie moduły dodają swoje wyjścia do jednego **residual stream**, to również **parametry** można rzutować na słownik i czytać jako listy tokenów. Odwrotność $E$ zastępuje się $E^T$ (lepiej współpracuje z obcinaniem do top-k), biasy i LayerNorm się pomija.

- **FF**: klucze (wiersze $W_{in}$) i wartości (wiersze $W_{out}$) to wektory w $\mathbb{R}^d$, więc $kE^T$ i $vE^T$ to rozkłady po słowniku.
- **Uwaga**: dla głowy $i$ macierze interakcji $W^i_{QK} = W^i_Q {W^i_K}^T$ i $W^i_{VO} = W^i_V W^i_O$ (obie $d \times d$, niezależne od wejścia). Po rzutowaniu:
  - $E\,W^i_{VO}\,E^T \in \mathbb{R}^{|V| \times |V|}$: element $(a, b)$ mówi, jak bardzo przyjście uwagi na token $a$ dopisuje do residual stream token $b$ (para *wejście → wyjście*; **załącznik C.1**),
  - $E\,W^i_{QK}\,E^T$: element $(a, b)$ mówi, jak bardzo token-zapytanie $a$ chce patrzeć na token-klucz $b$ (**załącznik C.2**).
- W załącznikach C.1/C.2 dla GPT-2 medium autorzy pokazują po **50 par o największych wartościach** dla ręcznie wybranych głów: pary podsłów i dokończenia słów (niski poziom LM), rodzaj (*his → her*), geografia (*Winnipeg → Manitoba*), pisownia brytyjska, słowa pokrewne tematycznie. Oznaczenia: `*` wykluczone pary $(x, x)$, `**` tylko tokeny występujące w korpusie.
- Ograniczenie: rama zakłada liniową głowę LM (GPT-2 ma $E^T$). BERT ma nieliniową głowę MLM, więc projekcja jest dla niego przybliżeniem (autorzy i tak pokazują, że działa przy wyrównywaniu warstw różnych BERT-ów).

## Przygotowanie

In [1]:
import json
from collections import defaultdict
from os.path import commonprefix
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
import torch
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer

pio.renderers.default = "plotly_mimetype+notebook_connected"
pd.set_option("display.max_colwidth", 300)
torch.set_grad_enabled(False)

MODELS = {"BERT (HerBERT-base)": "allegro/herbert-base-cased", "GPT-2 (polish-gpt2-medium)": "sdadas/polish-gpt2-medium"}
CORPUS = "skladnica"
MIN_CONTEXTS, MAX_CONTEXTS = 5, 20
N_WORDS, N_SENTS, N_BASELINE = 1000, 500, 1000
MAX_PER_SENSE = 15
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
COLORS = dict(zip(MODELS, ["#1f77b4", "#d62728"]))
CACHE = Path("cache")  # wyniki ciężkich obliczeń; usuń katalog, żeby policzyć od nowa
CACHE.mkdir(exist_ok=True)
print(DEVICE)

C:\Users\user\Desktop\SZT II ST\sem2\jezyk naturalny\korpus-nlp\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cuda


## Dane: Składnica z `wsd_polish_datasets`

Repozytorium używa skryptu ładującego, którego nowe wersje `datasets` nie uruchamiają, więc pobieramy plik JSONL bezpośrednio. Każda linia to dokument z listą zdań. Zdanie ma `text`, `tokens` (forma, lemat, część mowy, pozycja znakowa) i `wsd` (sens z plWordNetu dla części tokenów).

In [2]:
path = hf_hub_download("clarin-knext/wsd_polish_datasets", f"data/{CORPUS}_sentences.jsonl", repo_type="dataset")
sents = [s for line in open(path, encoding="utf-8") for s in json.loads(line)["sentences"]]

rows = []
for si, s in enumerate(sents):
    senses = {w["index"]: w["pl_sense"] for w in s.get("wsd") or []}
    for t in s["tokens"]:
        if t["pos"] != "interp":
            start, end = t["position"]
            rows.append({"sent": si, "tok": t["index"], "start": start, "end": end, "word": t["orth"].lower(),
                         "lemma": t["lemma"], "pos": t["pos"], "sense": senses.get(t["index"])})
occ = pd.DataFrame(rows)
assert all(sents[r.sent]["text"][r.start:r.end].lower() == r.word for r in occ.sample(2000, random_state=SEED).itertuples())

n_ctx = occ.drop_duplicates(["word", "sent"]).groupby("word").size()
pd.Series({"zdania": len(sents), "słowa (bez interpunkcji)": len(occ), "formy wyrazowe": occ.word.nunique(),
           f"formy w ≥ {MIN_CONTEXTS} kontekstach": (n_ctx >= MIN_CONTEXTS).sum(),
           "wystąpienia z sensem": occ.sense.notna().sum(), "lematy z sensem": occ[occ.sense.notna()].lemma.nunique()}).to_frame("liczba")

,liczba
zdania,11511
słowa (bez interpunkcji),113195
formy wyrazowe,32836
formy w ≥ 5 kontekstach,3015
wystąpienia z sensem,40525
lematy z sensem,9712


### Próbki do miar

Tak jak w artykule:
- **self-similarity i MEV**: 1000 losowych form wyrazowych występujących w co najmniej 5 różnych zdaniach (po jednym wystąpieniu na zdanie, maks. 20 kontekstów na słowo),
- **intra-sentence similarity**: 500 losowych zdań (co najmniej 5 słów),
- **baseline anizotropii**: 1000 losowych wystąpień słów, cosinus liczony tylko dla par z różnych zdań.

Dodatkowo **sensy**: lematy, które mają co najmniej 2 sensy z co najmniej 3 wystąpieniami każdy (bez sensów wielowyrazowych typu *być_w_mocy*).

In [3]:
rng = np.random.default_rng(SEED)
ctx = occ.drop_duplicates(["word", "sent"]).sample(frac=1, random_state=SEED)
eligible = n_ctx[n_ctx >= MIN_CONTEXTS].index.to_numpy()
self_occ = ctx[ctx.word.isin(rng.choice(eligible, min(N_WORDS, len(eligible)), replace=False))].groupby("word").head(MAX_CONTEXTS)
sent_len = occ.groupby("sent").size()
intra_occ = occ[occ.sent.isin(rng.choice(sent_len[sent_len >= 5].index, N_SENTS, replace=False))]
base_occ = occ.sample(N_BASELINE, random_state=SEED)

sense_occ = occ[occ.sense.str.rsplit(".", n=2).str[0] == occ.lemma].drop_duplicates(["lemma", "sent"])
sense_occ = sense_occ.sample(frac=1, random_state=SEED).groupby("sense").head(MAX_PER_SENSE)
good = sense_occ.groupby("sense").filter(lambda g: len(g) >= 3)
good = good.groupby("lemma").filter(lambda g: g.sense.nunique() >= 2)
sense_occ = good

needed = pd.concat([self_occ, intra_occ, base_occ, sense_occ]).drop_duplicates(["sent", "tok"]).reset_index(drop=True)
row_of = {k: i for i, k in enumerate(zip(needed.sent, needed.tok))}
rows_for = lambda frame: np.array([row_of[k] for k in zip(frame.sent, frame.tok)])

self_groups = [rows_for(g) for _, g in self_occ.groupby("word")]
intra_groups = [rows_for(g) for _, g in intra_occ.groupby("sent")]
base_rows, base_sent = rows_for(base_occ), base_occ.sent.to_numpy()
sense_groups = [(rows_for(g), pd.factorize(g.sense)[0]) for _, g in sense_occ.groupby("lemma")]
mev_base_rows = [rng.choice(base_rows, len(g), replace=False) for g in self_groups]

print(f"słowa: {len(self_groups)}, wystąpienia: {len(self_occ)}; zdania intra: {len(intra_groups)}; "
      f"lematy wieloznaczne: {len(sense_groups)} ({len(sense_occ)} wystąpień); razem wektorów: {len(needed)}")

słowa: 1000, wystąpienia: 10211; zdania intra: 500; lematy wieloznaczne: 408 (7044 wystąpień); razem wektorów: 21499


### Reprezentacje

Wektor słowa w warstwie $\ell$ to średnia wektorów jego podsłów (dopasowanych po pozycjach znakowych). Warstwa 0 to osadzenia wejściowe (token + pozycja).
GPT-2 dostaje na początku token `<s>`: pierwsza pozycja w GPT-2 działa jak „ujście uwagi” (*attention sink*) i ma ogromną normę, więc bez `<s>` zaburzałaby pierwsze słowo każdego zdania. W HerBERT tokeny specjalne `<s>`, `</s>` są dodawane standardowo i nie są brane pod uwagę.

In [4]:
def extract(name):
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModel.from_pretrained(name).to(DEVICE).eval()
    bos = [tok.bos_token_id] if model.config.model_type == "gpt2" else []
    R = np.zeros((len(needed), model.config.num_hidden_layers + 1, model.config.hidden_size), np.float16)
    for si, g in tqdm(needed.groupby("sent"), desc=name):
        enc = tok(sents[si]["text"], return_offsets_mapping=True, truncation=True, max_length=512)
        offs = np.array([(0, 0)] * len(bos) + enc["offset_mapping"])
        hs = model(torch.tensor([bos + enc["input_ids"]], device=DEVICE), output_hidden_states=True).hidden_states
        H = torch.stack(hs)[:, 0]
        M = (offs[:, 0][None] < g.end.to_numpy()[:, None]) & (offs[:, 1][None] > g.start.to_numpy()[:, None])
        assert M.any(1).all(), sents[si]["text"]
        M = torch.tensor(M / M.sum(1, keepdims=True), dtype=torch.float32, device=DEVICE)
        R[g.index] = torch.einsum("nt,ltd->nld", M, H).cpu().numpy()
    del model
    torch.cuda.empty_cache()
    return R

### Miary

In [5]:
def unit(X):
    return X / X.norm(dim=-1, keepdim=True)


def mean_offdiag(S):
    n = len(S)
    return ((S.sum() - S.diagonal().sum()) / (n * n - n)).item()


def mev(X):
    s = torch.linalg.svdvals(X) ** 2
    return (s[0] / s.sum()).item()


def layer_metrics(R, layer, standardize=False):
    X = torch.from_numpy(R[:, layer]).to(DEVICE).float()
    if standardize:
        X = (X - X.mean(0)) / X.std(0)
    Xn = unit(X)
    B = Xn[base_rows]
    diff = torch.from_numpy(base_sent[:, None] != base_sent[None, :]).to(DEVICE)
    baseline = (B @ B.T)[diff].mean().item()
    self_sim = np.array([mean_offdiag(Xn[g] @ Xn[g].T) for g in self_groups])
    intra = np.mean([(Xn[g] @ unit(X[g].mean(0))).mean().item() for g in intra_groups])
    mev_raw = np.array([mev(X[g]) for g in self_groups])
    mev_base = np.array([mev(X[g]) for g in mev_base_rows])

    same, other, hit, n = [], [], 0, 0
    for g, y in sense_groups:
        S = Xn[g] @ Xn[g].T
        eq = torch.from_numpy(y[:, None] == y[None, :]).to(DEVICE)
        off = ~torch.eye(len(g), dtype=torch.bool, device=DEVICE)
        same.append(S[eq & off].mean().item())
        other.append(S[~eq].mean().item())
        nn_idx = S.masked_fill(~off, -2).argmax(1).cpu().numpy()
        hit += (y[nn_idx] == y).sum()
        n += len(g)
    return {"wariant": "standaryzowane" if standardize else "surowe", "warstwa": layer, "anizotropia (baseline)": baseline,
            "self-similarity": self_sim.mean(), "self-similarity (skorygowane)": self_sim.mean() - baseline,
            "intra-sentence (surowe)": intra, "intra-sentence (skorygowane)": intra - baseline,
            "MEV (surowe)": mev_raw.mean(), "MEV (skorygowane)": (mev_raw - mev_base).mean(),
            "cos: ten sam sens": np.mean(same), "cos: inny sens": np.mean(other), "1-NN sensu (accuracy)": hit / n,
            "_self_sim_words": self_sim - baseline, **rogue_share(Xn[base_rows])}


def rogue_share(Xn, top=(1, 5)):
    # udział wymiarów o największym średnim wkładzie w cosinus losowych par
    i, j = torch.triu_indices(len(Xn), len(Xn), 1, device=DEVICE)
    contrib = (Xn[i] * Xn[j]).mean(0).abs().sort(descending=True).values
    return {f"_top-{k} wymiarów": (contrib[:k].sum() / contrib.sum()).item() for k in top}


def majority_baseline():
    hit, n = 0, 0
    for _, y in sense_groups:
        counts = np.bincount(y)
        for label in y:
            c = counts.copy()
            c[label] -= 1
            hit += c.argmax() == label
            n += 1
    return hit / n

In [6]:
results, word_self_sim = [], {}
for model_name, hf_name in MODELS.items():
    cache = CACHE / f"miary_{hf_name.replace('/', '_')}.pkl"
    if cache.exists():
        per_layer = pd.read_pickle(cache)
    else:
        R = extract(hf_name)
        per_layer = [layer_metrics(R, layer, standardize) for standardize in (False, True)
                     for layer in tqdm(range(R.shape[1]), desc="miary")]
        pd.to_pickle(per_layer, cache)
        del R
    L = max(m["warstwa"] for m in per_layer)
    for variant in ("surowe", "standaryzowane"):
        word_self_sim[model_name, variant] = np.stack([m["_self_sim_words"] for m in per_layer if m["wariant"] == variant])
    results += [{"model": model_name, "głębokość": m["warstwa"] / L, **m} for m in per_layer]
metrics = pd.DataFrame(results).drop(columns="_self_sim_words")
rogue_cols = [c for c in metrics if c.startswith("_top")]
rogue = metrics[metrics.wariant == "surowe"][["model", "warstwa", "anizotropia (baseline)", *rogue_cols]].rename(
    columns={c: "udział " + c[1:] for c in rogue_cols})
metrics = metrics.drop(columns=rogue_cols)
words_order = [w for w, _ in self_occ.groupby("word")]
SENSE_MAJORITY = majority_baseline()
metrics.drop(columns="głębokość").set_index(["model", "wariant", "warstwa"]).round(3)

anizotropia (baseline)  \
model                      wariant        warstwa                           
BERT (HerBERT-base)        surowe         0                         0.059   
                                          1                         0.283   
                                          2                         0.569   
                                          3                         0.618   
                                          4                         0.583   
...                                                                   ...   
GPT-2 (polish-gpt2-medium) standaryzowane 20                        0.004   
                                          21                        0.004   
                                          22                        0.004   
                                          23                        0.004   
                                          24                        0.004   

                                                   self-similarity  \
model                      wariant        warstwa                    
BERT (HerBERT-base)        surowe         0                  0.842   
                                          1                  0.790   
                                          2                  0.768   
                                          3                  0.781   
                                          4                  0.771   
...                                                            ...   
GPT-2 (polish-gpt2-medium) standaryzowane 20                 0.504   
                                          21                 0.491   
                                          22                 0.476   
                                          23                 0.463   
                                          24                 0.450   

                                                   self-similarity (skorygowane)  \
model                      wariant        warstwa                                  
BERT (HerBERT-base)        surowe         0                                0.783   
                                          1                                0.507   
                                          2                                0.199   
                                          3                                0.163   
                                          4                                0.187   
...                                                                          ...   
GPT-2 (polish-gpt2-medium) standaryzowane 20                               0.500   
                                          21                               0.488   
                                          22                               0.472   
                                          23                               0.459   
                                          24                               0.447   

                                                   intra-sentence (surowe)  \
model                      wariant        warstwa                            
BERT (HerBERT-base)        surowe         0                          0.405   
                                          1                          0.607   
                                          2                          0.785   
                                          3                          0.810   
                                          4                          0.792   
...                                                                    ...   
GPT-2 (polish-gpt2-medium) standaryzowane 20                         0.521   
                                          21                         0.521   
                                          22                         0.523   
                                          23                         0.525   
                                          24                         0.525   

                              

Warstwy obu modeli zestawiamy według **względnej głębokości** (warstwa / liczba warstw), bo HerBERT ma 12, a polish-gpt2-medium 24 warstwy.

In [7]:
def plot(cols, title, yaxis="", variants=("surowe", "standaryzowane"), **kw):
    long = metrics[metrics.wariant.isin(variants)].melt(id_vars=["model", "wariant", "warstwa", "głębokość"],
                                                        value_vars=cols, var_name="miara")
    fig = px.line(long, x="głębokość", y="value", color="model", line_dash="miara" if len(cols) > 1 else None,
                  facet_col="wariant" if len(variants) > 1 else None, markers=True, color_discrete_map=COLORS,
                  hover_data={"warstwa": True}, title=title,
                  labels={"value": yaxis, "głębokość": "względna głębokość (0 = osadzenia wejściowe)"},
                  width=1050, height=450, **kw)
    fig.show()

## 1a. Anizotropia

In [8]:
plot(["anizotropia (baseline)"], "Średni cosinus losowych słów z różnych zdań (0 = izotropia)", "cos", variants=("surowe",))

Skąd anizotropia GPT-2? Tradycyjne wyjaśnienie (Ethayarajh, Gao i in. 2019) mówi o „degeneracji” przestrzeni przy uczeniu LM. Timkey i van Schijndel (2021) pokazali, że w GPT-2 cosinus jest zdominowany przez **kilka „zbuntowanych” wymiarów** (*rogue dimensions*) o ogromnej wariancji. Sprawdzamy, jaką część iloczynu skalarnego losowych par wnosi 1 i 5 największych wymiarów (według średniego wkładu $|E[x_i y_i]|$).

In [9]:
rogue.set_index(["model", "warstwa"]).round(3)

anizotropia (baseline)  \
model                      warstwa                           
BERT (HerBERT-base)        0                         0.059   
                           1                         0.283   
                           2                         0.569   
                           3                         0.618   
                           4                         0.583   
                           5                         0.598   
                           6                         0.600   
                           7                         0.603   
                           8                         0.644   
                           9                         0.671   
                           10                        0.656   
                           11                        0.723   
                           12                        0.741   
GPT-2 (polish-gpt2-medium) 0                         0.125   
                           1                         0.725   
                           2                         0.724   
                           3                         0.715   
                           4                         0.685   
                           5                         0.648   
                           6                         0.626   
                           7                         0.608   
                           8                         0.590   
                           9                         0.573   
                           10                        0.563   
                           11                        0.562   
                           12                        0.563   
                           13                        0.568   
                           14                        0.572   
                           15                        0.573   
                           16                        0.560   
                           17                        0.560   
                           18                        0.572   
                           19                        0.580   
                           20                        0.607   
                           21                        0.637   
                           22                        0.683   
                           23                        0.698   
                           24                        0.227   

                                    udział top-1 wymiarów  \
model                      warstwa                          
BERT (HerBERT-base)        0                        0.039   
                           1                        0.721   
                           2                        0.889   
                           3                        0.911   
                           4                        0.892   
                           5                        0.892   
                           6                        0.896   
                           7                        0.900   
                           8                        0.897   
                           9                        0.858   
                           10                       0.781   
                           11                       0.877   
                           12                       0.907   
GPT-2 (polish-gpt2-medium) 0                        0.050   
                           1                        0.297   
                           2                        0.286   
                           3                        0.270   
                           4                        0.277   
                           5                        0.284   
                           6                        0.301   
                           7                        0.314   
                           8                        0.312   
                           9                        0.300   
                           10 

W HerBERT od warstwy 2 **jeden wymiar daje ok. 90% cosinusa** losowych par. Przy bliższym przyjrzeniu się (wymiar 588) ma on prawie zawsze wartość ok. −15 (pozostałe wymiary mają odchylenie ok. 0,3), ale **zeruje się dla tokenu na drugiej pozycji zdania i dla `</s>`**. To artefakt pozycyjny: słowo, które wypadło na drugiej pozycji, ma wektor „odwrócony” w dominującym wymiarze. Jego cosinus z innymi wystąpieniami tego samego słowa spada wtedy poniżej cosinusa losowych słów (np. *teren*, *Tadeusz*, *trzecie* mają ujemne skorygowane self-similarity). W GPT-2 5 wymiarów daje 85–93% cosinusa.

Dlatego każdą miarę liczymy w dwóch wariantach:
- **surowe**: dokładnie jak w artykule,
- **standaryzowane**: każdy wymiar w danej warstwie sprowadzony do średniej 0 i odchylenia 1 (z-score po wszystkich zebranych wektorach), czyli poprawka zalecana przez Timkeya i van Schijndela. Usuwa wspólny kierunek (więc baseline anizotropii jest ok. 0) i nie pozwala kilku wymiarom zdominować cosinusa.

## 1b. Zależność od kontekstu (context-specificity)

In [10]:
plot(["self-similarity (skorygowane)"], "Self-similarity skorygowane o anizotropię (niżej = bardziej kontekstowe)", "SelfSim*")
plot(["self-similarity", "anizotropia (baseline)"], "Self-similarity surowe vs baseline", "cos", variants=("surowe",))

In [11]:
plot(["intra-sentence (skorygowane)"], "Intra-sentence similarity skorygowane o anizotropię", "IntraSim*")

In [12]:
plot(["MEV (skorygowane)", "MEV (surowe)"], "MEV: ile wariancji wystąpień słowa wyjaśnia jeden wektor statyczny", "MEV")

### Które słowa są najbardziej kontekstowe?

Średnie skorygowane self-similarity po warstwach 1..L (bez warstwy wejściowej), wariant standaryzowany (w surowym HerBERT ranking psuje artefakt drugiej pozycji). Artykuł: najbardziej kontekstowe są słowa funkcyjne.

In [13]:
pos_of = occ.groupby("word").pos.agg(lambda s: s.mode()[0])
tables = {}
for (model_name, variant), S in word_self_sim.items():
    t = pd.DataFrame({"słowo": words_order, "SelfSim*": S[1:].mean(0)})
    t["część mowy"] = t["słowo"].map(pos_of)
    t["konteksty"] = t["słowo"].map(n_ctx).clip(upper=MAX_CONTEXTS)
    tables[model_name, variant] = t.sort_values("SelfSim*")
std_tables = {m: t for (m, v), t in tables.items() if v == "standaryzowane"}
print("najbardziej kontekstowe (najniższe SelfSim*)")
display(pd.concat({m: t.head(15).reset_index(drop=True) for m, t in std_tables.items()}, axis=1).round(3))
print("najmniej kontekstowe")
display(pd.concat({m: t.tail(10).iloc[::-1].reset_index(drop=True) for m, t in std_tables.items()}, axis=1).round(3))

najbardziej kontekstowe (najniższe SelfSim*)


BERT (HerBERT-base)                                \
                 słowo SelfSim* część mowy konteksty   
0            trzeciego    0.219        adj         6   
1              głównym    0.220        adj         6   
2              słyszał    0.245       verb         5   
3            wspaniały    0.258        adj         5   
4          rozwiązania    0.260       noun         5   
5                   to    0.266       noun        20   
6                teren    0.277       noun         5   
7              trzecie    0.280        adj         5   
8                   na    0.281       prep        20   
9                marek    0.283       noun         9   
10              drugim    0.284        adj         9   
11             tadeusz    0.296       noun         5   
12              dostał    0.296       verb         7   
13              opinii    0.296       noun         6   
14                 tym    0.297       noun        20   

   GPT-2 (polish-gpt2-medium)                                
                        słowo SelfSim* część mowy konteksty  
0                           ś    0.271       aglt        20  
1                         nim    0.359     ppron3        20  
2                          to    0.359       noun        20  
3                   wszystkim    0.366       noun        20  
4                         tym    0.380       noun        20  
5                         one    0.381     ppron3        20  
6                       czasu    0.383       noun        20  
7                        niej    0.387     ppron3        20  
8                       razem    0.387       noun        20  
9                         ona    0.388     ppron3        20  
10                     moment    0.398       noun        12  
11                       mnie    0.400    ppron12        20  
12                      kolei    0.400       noun        13  
13                         na    0.402       prep        20  
14                      życie    0.408       noun        20

najmniej kontekstowe


BERT (HerBERT-base)                                \
                słowo SelfSim* część mowy konteksty   
0         naciśnięcie    0.993       verb         5   
1            zabranie    0.944       verb         7   
2         pozbawienia    0.898       noun         7   
3           dziennego    0.853        adj         5   
4                godz    0.848       noun        11   
5                 mld    0.824       noun         8   
6                  br    0.823       noun        11   
7                  tp    0.805       noun         5   
8            minutach    0.800       noun         8   
9               stołu    0.794       noun         7   

  GPT-2 (polish-gpt2-medium)                                
                       słowo SelfSim* część mowy konteksty  
0                naciśnięcie    0.994       verb         5  
1                   zabranie    0.986       verb         7  
2                pozbawienia    0.906       noun         7  
3                przepraszam    0.903       verb        11  
4                    grudniu    0.901       noun         5  
5                  przyszłym    0.862        adj        11  
6                      jeśli    0.858       comp        20  
7                        mld    0.857       noun         8  
8                   rzecznik    0.851       noun         8  
9                   minutach    0.849       noun         8

In [14]:
FUNCTION_POS = {"prep", "conj", "comp", "qub", "pron", "ppron12", "ppron3", "siebie", "aglt"}
rows = []
for (model_name, variant), t in tables.items():
    t = t.assign(grupa=np.where(t["część mowy"].isin(FUNCTION_POS), "funkcyjne", "treściowe"))
    rows.append({"model": model_name, "wariant": variant,
                 **t.groupby("grupa")["SelfSim*"].mean().rename(lambda g: f"SelfSim*: {g}"),
                 "liczba funkcyjnych": (t.grupa == "funkcyjne").sum()})
pd.DataFrame(rows).set_index(["model", "wariant"]).round(3)

SelfSim*: funkcyjne  \
model                      wariant                               
BERT (HerBERT-base)        surowe                        0.163   
                           standaryzowane                0.508   
GPT-2 (polish-gpt2-medium) surowe                        0.211   
                           standaryzowane                0.555   

                                           SelfSim*: treściowe  \
model                      wariant                               
BERT (HerBERT-base)        surowe                        0.171   
                           standaryzowane                0.527   
GPT-2 (polish-gpt2-medium) surowe                        0.237   
                           standaryzowane                0.620   

                                           liczba funkcyjnych  
model                      wariant                             
BERT (HerBERT-base)        surowe                          61  
                           standaryzowane                  61  
GPT-2 (polish-gpt2-medium) surowe                          61  
                           standaryzowane                  61

### Sensy (rozszerzenie dzięki anotacji WSD)

Dla lematów wieloznacznych porównujemy cosinus między wystąpieniami **tego samego sensu** i **różnych sensów** oraz dokładność klasyfikatora 1-NN (sens najbliższego innego wystąpienia tego samego lematu, z innego zdania). Punkt odniesienia: najczęstszy sens wśród pozostałych wystąpień lematu.

In [15]:
metrics["różnica: ten sam − inny sens"] = metrics["cos: ten sam sens"] - metrics["cos: inny sens"]
plot(["różnica: ten sam − inny sens"], "O ile wystąpienia tego samego sensu są bliżej siebie niż wystąpienia różnych sensów", "Δ cos")
fig = px.line(metrics, x="głębokość", y="1-NN sensu (accuracy)", color="model", facet_col="wariant", markers=True,
              color_discrete_map=COLORS, hover_data={"warstwa": True},
              title=f"Rozróżnianie sensów 1-NN (najczęstszy sens: {SENSE_MAJORITY:.3f})", width=1050, height=450)
fig.add_hline(SENSE_MAJORITY, line_dash="dot", annotation_text="najczęstszy sens")
fig.show()
print(len(sense_groups), "lematów,", sum(len(g) for g, _ in sense_groups), "wystąpień")

408 lematów, 7044 wystąpień


### Podsumowanie liczbowe

In [16]:
def summary(frame):
    inner = frame[(frame.warstwa > 0)]
    best = frame.loc[frame["1-NN sensu (accuracy)"].idxmax()]
    return pd.Series({
        "anizotropia: warstwa 0": frame.iloc[0]["anizotropia (baseline)"],
        "anizotropia: min / max (warstwy ≥ 1)": f"{inner['anizotropia (baseline)'].min():.3f} / {inner['anizotropia (baseline)'].max():.3f}",
        "anizotropia: ostatnia warstwa": frame.iloc[-1]["anizotropia (baseline)"],
        "SelfSim*: warstwa 1 → ostatnia": f"{frame.iloc[1]['self-similarity (skorygowane)']:.3f} → {frame.iloc[-1]['self-similarity (skorygowane)']:.3f}",
        "IntraSim*: zakres (warstwy ≥ 1)": f"{inner['intra-sentence (skorygowane)'].min():.3f} – {inner['intra-sentence (skorygowane)'].max():.3f}",
        "MEV*: maks. (warstwy ≥ 1)": inner["MEV (skorygowane)"].max(),
        "MEV surowe: średnio (warstwy ≥ 1)": inner["MEV (surowe)"].mean(),
        "1-NN sensu: najlepsza warstwa": f"{best['1-NN sensu (accuracy)']:.3f} (warstwa {int(best['warstwa'])})",
    })


pd.DataFrame({key: summary(g.reset_index(drop=True)) for key, g in metrics.groupby(["model", "wariant"], sort=False)})

BERT (HerBERT-base)                      \
                                                  surowe      standaryzowane   
anizotropia: warstwa 0                          0.058679            0.012413   
anizotropia: min / max (warstwy ≥ 1)       0.283 / 0.741       0.005 / 0.011   
anizotropia: ostatnia warstwa                   0.741134            0.006585   
SelfSim*: warstwa 1 → ostatnia             0.507 → 0.118       0.780 → 0.571   
IntraSim*: zakres (warstwy ≥ 1)            0.145 – 0.324       0.378 – 0.481   
MEV*: maks. (warstwy ≥ 1)                         0.3975            0.645282   
MEV surowe: średnio (warstwy ≥ 1)               0.869963            0.651065   
1-NN sensu: najlepsza warstwa         0.609 (warstwa 10)  0.620 (warstwa 10)   

                                     GPT-2 (polish-gpt2-medium)  \
                                                         surowe   
anizotropia: warstwa 0                                 0.125063   
anizotropia: min / max (warstwy ≥ 1)              0.227 / 0.725   
anizotropia: ostatnia warstwa                          0.227364   
SelfSim*: warstwa 1 → ostatnia                    0.237 → 0.348   
IntraSim*: zakres (warstwy ≥ 1)                   0.148 – 0.439   
MEV*: maks. (warstwy ≥ 1)                              0.305082   
MEV surowe: średnio (warstwy ≥ 1)                       0.85849   
1-NN sensu: najlepsza warstwa                0.590 (warstwa 14)   

                                                          
                                          standaryzowane  
anizotropia: warstwa 0                           0.01283  
anizotropia: min / max (warstwy ≥ 1)       0.004 / 0.012  
anizotropia: ostatnia warstwa                   0.003895  
SelfSim*: warstwa 1 → ostatnia             0.867 → 0.447  
IntraSim*: zakres (warstwy ≥ 1)            0.351 – 0.521  
MEV*: maks. (warstwy ≥ 1)                       0.700503  
MEV surowe: średnio (warstwy ≥ 1)               0.678391  
1-NN sensu: najlepsza warstwa         0.594 (warstwa 17)

## Podsumowanie części 1

**Anizotropia (1a): potwierdzona, ale z dwoma zastrzeżeniami.**
- W obu modelach osadzenia wejściowe są prawie izotropowe (średni cosinus losowych słów 0,06 w HerBERT, 0,13 w GPT-2), a **wszystkie warstwy kontekstowe są silnie anizotropowe**. HerBERT rośnie z 0,28 (warstwa 1) do 0,74 (warstwa 12). W GPT-2 anizotropia skacze do 0,73 już w warstwie 1, spada do ok. 0,56 w środku i znów rośnie do 0,70 w warstwie 23. Tak jak w artykule: kontekstualizacji towarzyszy anizotropia.
- Ostatnia warstwa GPT-2 (24) ma tylko 0,23, a nie prawie 1 jak w artykule. `transformers` zwraca jako ostatni stan ukryty wyjście **po końcowym LayerNorm** (`ln_f`), który centruje i skaluje wektory. Warstwa 23, przed `ln_f`, zachowuje się „po ethayarajhowemu”.
- **Anizotropia to w dużej mierze kilka wymiarów.** W HerBERT od warstwy 2 jeden wymiar daje ok. 90% cosinusa losowych par, w GPT-2 pięć wymiarów daje 85–93%. Ten wymiar w HerBERT ma przy tym artefakt pozycyjny (zeruje się na 2. pozycji zdania). Surowe cosinusy w tych modelach mierzą więc głównie kilka wymiarów, a nie znaczenie. Stąd drugi, standaryzowany wariant miar.

**Zależność od kontekstu (1b).**
- **Wyższe warstwy są bardziej kontekstowe.** Standaryzowane self-similarity:
  - GPT-2: spada prawie monotonicznie z 0,87 (warstwa 1) do 0,45 (warstwa 24);
  - HerBERT: spada z 0,78 do minimum 0,42 w warstwach 7–8, potem lekko rośnie do 0,57 w warstwie 12. Ostatnia warstwa BERT-a jest „ściągana” w stronę zadania MLM, czyli przewidywania samego słowa;
  - w wariancie surowym trend jest ten sam dla HerBERT (0,51 → 0,07–0,12). W GPT-2 przez rogue dimensions jest płaski aż do ok. warstwy 18, potem spada do 0,13.
- **Najbardziej kontekstowe są słowa funkcyjne.** W GPT-2 (standaryzacja) najniższe self-similarity mają zaimki osobowe (*nim, one, niej, ona, mnie*), *to, tym, na*. Średnio słowa funkcyjne mają 0,555, a treściowe 0,620. W HerBERT różnica jest mała (0,508 vs 0,527), a ranking wciąż psuje artefakt drugiej pozycji (*teren, Tadeusz, trzecie*). Najmniej kontekstowe są skróty i słowa ze stałych połączeń (*mld, godz., pozbawienia [wolności], zabranie [głosu]*). To zgadza się z tezą autora: o zmienności decyduje różnorodność kontekstów, a nie liczba sensów.
- **Intra-sentence similarity zależy od wariantu.**
  - Surowo HerBERT zachowuje się jak BERT w artykule: słowa jednego zdania oddalają się od siebie w górnych warstwach (0,32 → 0,15), ale są bliżej niż losowe słowa;
  - w GPT-2 skorygowane IntraSim nie jest bliskie 0, jak w artykule, tylko wynosi 0,15–0,26;
  - po standaryzacji w obu modelach IntraSim **rośnie** z głębokością (HerBERT 0,33 → 0,48, GPT-2 0,34 → 0,52), czyli słowa jednego zdania upodabniają się do siebie, jak w ELMo.
  - Wniosek artykułu o „sposobie przejawiania się kontekstowości” okazuje się więc wrażliwy na kilka dominujących wymiarów.
- **MEV.** Udział wariancji wyjaśnianej przez jeden wektor statyczny maleje z głębokością (skorygowany, surowo: HerBERT 0,40 → 0,08–0,10, GPT-2 ok. 0,22 → 0,11). Nie schodzi jednak poniżej 5%, jak w artykule. Mamy tylko 5–20 kontekstów na słowo, a przy małej liczbie wektorów pierwsza składowa z definicji wyjaśnia dużą część wariancji. Odejmujemy co prawda baseline z tą samą liczbą losowych wektorów, ale wartości bezwzględne nie są porównywalne z artykułem. Porównywalny jest trend: im wyżej, tym gorzej wektor statyczny zastępuje kontekstowe.

**Sensy (rozszerzenie dzięki WSD).**
- Wystąpienia tego samego sensu są we wszystkich warstwach bliżej siebie niż wystąpienia różnych sensów. Po standaryzacji różnica rośnie z ok. 0,03 (warstwa 0) do ok. 0,07 w górnych warstwach obu modeli.
- 1-NN po sensie daje 0,50 w warstwie 0 i do **0,62** w HerBERT (warstwa 10) i **0,59** w GPT-2 (warstwy 17–21), przy 0,44 dla najczęstszego sensu.
- Kontekstowe reprezentacje niosą więc informację o sensie, i to głównie w górnych warstwach, ale daleko im do „jednego punktu na sens”. Różnica 0,07 cosinusa jest mała wobec ogólnej zmienności wystąpień, co też zgadza się z wnioskiem artykułu: modele nie przypisują słowu jednego z kilku wektorów-sensów.
- HerBERT (dwukierunkowy) radzi sobie lepiej niż GPT-2, który widzi tylko lewy kontekst.

## 2. Parameter Projection

Dla każdej głowy $i$ w warstwie $\ell$ liczymy macierze $|V| \times |V|$:
- $E\,W^{i}_{VO}\,E^T$, gdzie $W_{VO} = W_V W_O$: para *(token, na który patrzy głowa → token, który głowa dopisuje do residual stream)* (C.1),
- $E\,W^{i}_{QK}\,E^T$, gdzie $W_{QK} = W_Q W_K^T$: para *(token-zapytanie → token-klucz, na który chce patrzeć)* (C.2),

i wybieramy 50 par o największych wartościach. Pełna macierz $51\,200 \times 51\,200$ ma 2,6 mld elementów (10 GB w float32), ale ma rząd co najwyżej $d/H = 64$, więc liczymy ją blokami z czynników $E W_V$ ($|V| \times 64$) i $W_O E^T$ ($64 \times |V|$), zostawiając w każdym bloku tylko najlepsze pary.

Tak jak u autorów: bez biasów i LayerNorm, $E$ to macierz osadzeń wejściowych (w GPT-2 wspólna z głową LM; w HerBERT także wspólna z dekoderem MLM).

Filtrowanie tokenów (odpowiednik `only_ascii`/`only_alnum` z kodu autorów): zostawiamy tokeny złożone z liter (`str.isalpha`, więc z polskimi znakami), co najmniej 2-literowe.
Bez tego listy zdominowałyby niepełne sekwencje bajtów UTF-8 (GPT-2 zapisuje *ą, ę, ł…* na poziomie bajtów) i znaki interpunkcyjne. Oznaczenia list jak w artykule:
- `*` wykluczone pary $(x, x)$, czyli ten sam token albo ta sama forma (bez wielkości liter i znaczników podsłów),
- `**` tylko tokeny, które występują w tokenizacji korpusu Składnica (odpowiednik filtra IMDB).

Zapis tokenów: GPT-2 oznacza **początek** słowa spacją, więc token bez spacji (kontynuację słowa) poprzedzamy `#` (konwencja autorów). HerBERT oznacza **koniec** słowa (`</w>`), więc token, który nie kończy słowa, ma na końcu `-`.

In [17]:
def attention_weights(model):
    if model.config.model_type == "gpt2":
        d, H = model.config.n_embd, model.config.n_head
        layers = [(*b.attn.c_attn.weight.split(d, dim=1), b.attn.c_proj.weight) for b in model.h]
        E = model.wte.weight
    else:
        d, H = model.config.hidden_size, model.config.num_attention_heads
        layers = [(l.attention.self.query.weight.T, l.attention.self.key.weight.T, l.attention.self.value.weight.T,
                   l.attention.output.dense.weight.T) for l in model.encoder.layer]
        E = model.embeddings.word_embeddings.weight
    hd = d // H
    heads = lambda W: W.reshape(d, H, hd).permute(1, 0, 2)
    Wq, Wk, Wv = (torch.stack([heads(l[i]) for l in layers]) for i in range(3))
    Wo = torch.stack([l[3].reshape(H, hd, d) for l in layers])
    return E.detach().float().to(DEVICE), Wq.detach().to(DEVICE), Wk.detach().to(DEVICE), Wv.detach().to(DEVICE), Wo.detach().to(DEVICE)


def token_table(tok, is_gpt):
    ids = np.arange(len(tok))
    raw = tok.convert_ids_to_tokens(ids.tolist())
    if is_gpt:
        text = [tok.convert_tokens_to_string([t]) for t in raw]
        core = [t.strip() for t in text]
        shown = [c if t.startswith(" ") else "#" + c for t, c in zip(text, core)]
    else:
        core = [t.replace("</w>", "") for t in raw]
        shown = [c if t.endswith("</w>") else c + "-" for t, c in zip(raw, core)]
    alpha = np.array([len(c) >= 2 and c.isalpha() for c in core])
    return pd.DataFrame({"core": core, "shown": shown, "alpha": alpha, "norm": [c.lower() for c in core]})

In [18]:
def top_pairs(A, B, allowed, group, k=50, chunk=2048):
    # najlepsze pary (wiersz, kolumna) macierzy A @ B ograniczonej do dozwolonych tokenów; wariant raw i bez par (x, x)
    idx = torch.from_numpy(np.flatnonzero(allowed)).to(DEVICE)
    Bc, gcol = B[:, idx], group[idx]
    found = {"raw": [], "noself": []}
    for s in range(0, len(idx), chunk):
        r = idx[s:s + chunk]
        S = A[r] @ Bc
        for kind in found:
            if kind == "noself":
                S = S.masked_fill(group[r][:, None] == gcol[None, :], -torch.inf)
            v, c = S.topk(k, dim=1)
            v, flat = v.flatten().topk(k)
            found[kind].append((v, r[flat // k], idx[c.flatten()[flat]]))
    out = {}
    for kind, parts in found.items():
        v, a, b = (torch.cat(x) for x in zip(*parts))
        order = v.topk(k).indices
        out[kind] = list(zip(a[order].tolist(), b[order].tolist(), v[order].tolist()))
    return out


class Projector:
    def __init__(self, hf_name):
        tok = AutoTokenizer.from_pretrained(hf_name)
        model = AutoModel.from_pretrained(hf_name)
        self.E, self.Wq, self.Wk, self.Wv, self.Wo = attention_weights(model)
        self.tt = token_table(tok, model.config.model_type == "gpt2")
        self.group = torch.from_numpy(pd.factorize(self.tt.norm)[0]).to(DEVICE)
        seen = np.zeros(len(self.tt), bool)
        for s in sents:
            seen[tok(s["text"], add_special_tokens=False)["input_ids"]] = True
        self.allowed = {False: self.tt.alpha.to_numpy(), True: self.tt.alpha.to_numpy() & seen}

    def pairs(self, kind, layer, h, corpus_only=False):
        if kind == "VO":
            A, B = self.E @ self.Wv[layer, h], self.Wo[layer, h] @ self.E.T
        else:
            A, B = self.E @ self.Wq[layer, h], (self.E @ self.Wk[layer, h]).T
        return top_pairs(A, B, self.allowed[corpus_only], self.group)

    def all_heads(self):
        L, H = self.Wq.shape[:2]
        return {(kind, layer, h): self.pairs(kind, layer, h)
                for layer in tqdm(range(L)) for h in range(H) for kind in ("VO", "QK")}

In [19]:
projectors, projections = {}, {}
for model_name, hf_name in MODELS.items():
    projectors[model_name] = p = Projector(hf_name)
    cache = CACHE / f"projekcje_{hf_name.replace('/', '_')}.pkl"
    if not cache.exists():
        pd.to_pickle(p.all_heads(), cache)
    projections[model_name] = (p.tt, pd.read_pickle(cache))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 28391.95it/s]


[transformers] BertModel LOAD REPORT from: allegro/herbert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.sso.sso_relationship.weight            | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.sso.sso_relationship.bias              | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

Loading weights:   5%|▌         | 15/292 [00:00<00:02, 120.04it/s]

Loading weights:  10%|▉         | 29/292 [00:00<00:02, 129.68it/s]

... (2 outputs left out)

Loading weights:  23%|██▎       | 67/292 [00:00<00:02, 91.79it/s]

Loading weights:  26%|██▋       | 77/292 [00:00<00:03, 62.70it/s]

Loading weights: 100%|██████████| 292/292 [00:00<00:00, 305.00it/s]


[transformers] GPT2Model LOAD REPORT from: sdadas/polish-gpt2-medium
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...23}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### Charakterystyka wszystkich głów

Żeby wybrać głowy do list (autorzy wybierali ręcznie spośród 384), liczymy dla każdej głowy proste wskaźniki z jej top-50 par:
- **kopiowanie**: odsetek par $(x, x)$ w liście bez wykluczeń (głowa „przepisuje” token, na który patrzy, albo patrzy na ten sam token),
- **morfologia**: odsetek par z listy `*`, w których oba tokeny mają wspólny początek ≥ 4 liter (odmiana / słowotwórstwo, np. *rok → roku*),
- **różnorodność**: liczba różnych tokenów w parach listy `*` (mała = lista zdominowana przez kilka tokenów).

In [20]:
def head_stats(model_name):
    tt, lists = projections[model_name]
    rows = []
    for (kind, layer, h), res in lists.items():
        raw, noself = res["raw"], res["noself"]
        morph = [len(commonprefix([tt.norm[a], tt.norm[b]])) >= 4 for a, b, _ in noself]
        rows.append({"model": model_name, "macierz": kind, "warstwa": layer, "głowa": h,
                     "kopiowanie": np.mean([tt.norm[a] == tt.norm[b] for a, b, _ in raw]),
                     "morfologia": np.mean(morph),
                     "różnorodność": len({x for a, b, _ in noself for x in (a, b)})})
    return pd.DataFrame(rows)


stats = pd.concat([head_stats(m) for m in MODELS])
stats["głębokość"] = stats.warstwa / stats.groupby("model").warstwa.transform("max")
stats["tercja"] = pd.cut(stats["głębokość"], [-0.01, 1 / 3, 2 / 3, 1], labels=["dolne", "środkowe", "górne"])
display(stats.groupby(["model", "macierz"])[["kopiowanie", "morfologia", "różnorodność"]].mean().round(3))
display(stats.pivot_table(index=["model", "macierz"], columns="tercja", values="morfologia", observed=True).round(3)
        .rename(columns=lambda c: f"morfologia: {c} warstwy"))
for col in ["kopiowanie", "morfologia"]:
    px.scatter(stats, x="głębokość", y=col, color="model", facet_col="macierz", color_discrete_map=COLORS, opacity=0.6,
               hover_data=["warstwa", "głowa", "różnorodność"], title=f"Głowy: {col} (top-50 par)", width=950, height=400).show()

kopiowanie  morfologia  różnorodność
model                      macierz                                      
BERT (HerBERT-base)        QK            0.025       0.006        50.847
                           VO            0.151       0.013        55.736
GPT-2 (polish-gpt2-medium) QK            0.069       0.053        54.747
                           VO            0.139       0.083        58.703

tercja                              morfologia: dolne warstwy  \
model                      macierz                              
BERT (HerBERT-base)        QK                           0.015   
                           VO                           0.007   
GPT-2 (polish-gpt2-medium) QK                           0.004   
                           VO                           0.015   

tercja                              morfologia: środkowe warstwy  \
model                      macierz                                 
BERT (HerBERT-base)        QK                              0.002   
                           VO                              0.013   
GPT-2 (polish-gpt2-medium) QK                              0.035   
                           VO                              0.079   

tercja                              morfologia: górne warstwy  
model                      macierz                             
BERT (HerBERT-base)        QK                           0.000  
                           VO                           0.020  
GPT-2 (polish-gpt2-medium) QK                           0.121  
                           VO                           0.154

In [21]:
def show(model_name, kind, layer, head, variant="noself", corpus_only=False, title=""):
    tt, lists = projections[model_name]
    res = projectors[model_name].pairs(kind, layer, head, corpus_only=True) if corpus_only else lists[kind, layer, head]
    mark = " ".join(m for m, on in (("*", variant == "noself"), ("**", corpus_only)) if on)
    print(f"{model_name} – {kind} – warstwa {layer}, głowa {head} {mark}  {title}")
    print(", ".join(f"({tt.shown[a]}, {tt.shown[b]})" for a, b, _ in res[variant]))
    print()


def find(model_name, kind, **conds):
    s = stats[(stats.model == model_name) & (stats.macierz == kind)]
    for col, (op, val) in conds.items():
        s = s[s[col] >= val] if op == ">=" else s[s[col] <= val]
    return s

Do przeglądania: głowy o najwyższym udziale par morfologicznych. Dowolną głowę można wypisać przez `show(model, "VO"/"QK", warstwa, głowa)`.

In [22]:
cols = ["model", "macierz", "warstwa", "głowa", "kopiowanie", "morfologia", "różnorodność"]
stats.sort_values("morfologia", ascending=False).groupby(["model", "macierz"]).head(5)[cols].round(2)

,model,macierz,warstwa,głowa,kopiowanie,morfologia,różnorodność
300,GPT-2 (polish-gpt2-medium),VO,9,6,1.00,0.92,79
716,GPT-2 (polish-gpt2-medium),VO,22,6,0.98,0.90,43
656,GPT-2 (polish-gpt2-medium),VO,20,8,0.66,0.90,39
439,GPT-2 (polish-gpt2-medium),QK,13,11,0.12,0.90,12
746,GPT-2 (polish-gpt2-medium),VO,23,5,1.00,0.90,44
564,GPT-2 (polish-gpt2-medium),VO,17,10,0.78,0.84,46
597,GPT-2 (polish-gpt2-medium),QK,18,10,0.68,0.66,69
749,GPT-2 (polish-gpt2-medium),QK,23,6,0.22,0.66,42
763,GPT-2 (polish-gpt2-medium),QK,23,13,0.12,0.62,15
67,BERT (HerBERT-base),QK,2,9,0.86,0.62,55


## C.1: macierze $W_{VO}$

Tak jak w artykule: ręcznie wybrane głowy, 50 par *(token, na który patrzy głowa → token dopisywany do residual stream)*, pogrupowane według tego, co głowa robi. Wszystkie listy są `*`, czyli bez par $(x, x)$.

GPT = `sdadas/polish-gpt2-medium`, warstwy 0–23.

### C.1.1 Niski poziom: odmiana i końcówki
Odpowiednik „Low-Level Language Modeling” z artykułu. W polszczyźnie zamiast dokańczania angielskich słów dominuje **odmiana**: głowa, która patrzy na formę wyrazu, dopisuje jego inne formy.

In [23]:
BERT, GPT = MODELS
show(GPT, "VO", 9, 6, title="(odmiana rzeczowników i przymiotników)")
show(GPT, "VO", 6, 3, title="(końcówki fleksyjne podsłów: #nicą → #nicami)")
show(GPT, "VO", 8, 2, title="(końcówki trybu przypuszczającego i 2. os. lm.)")

GPT-2 (polish-gpt2-medium) – VO – warstwa 9, głowa 6 *  (odmiana rzeczowników i przymiotników)
(rocznica, rocznicy), (certyfikat, certyfikaty), (kąpiel, kąpie), (obyczajów, obyczaje), (korupcji, korup), (płynów, płynu), (#burga, #burgu), (receptę, recepty), (prawną, prawna), (prawną, prawnej), (płuca, płuc), (powiet, powietrza), (płuc, płucach), (#dliwość, #dliwości), (atmos, atmosferę), (oferty, ofert), (#burgu, #burga), (#eria, #erii), (atmos, atmosfery), (hym, hymn), (handlowe, handlowym), (bieliznę, bielizny), (certyfika, certyfikaty), (obowiązkami, obowiązków), (regulaminem, Regulami), (korup, korupcji), (handlowy, handlowym), (napojów, napoju), (weterynaryj, wetery), (konkursów, konkursy), (alkohol, alkoholu), (obowiązkami, obowiązkach), (sądowe, sądowych), (#erii, #eria), (finansowego, finansowy), (medycy, medycyny), (czynsz, czynszu), (piersiową, piersiowej), (majątkowe, majątkowych), (mundur, munduru), (datą, datę), (rzemi, rzemiosła), (medycznej, medyczną), (medycyny, medycy)

### C.1.2 Rodzaj
Odpowiednik głowy *his → her*. W polskim rodzaj jest gramatyczny i przenika czasowniki, przymiotniki i zaimki: **rzeczownik lub forma żeńska → żeński zaimek** (*dziewczynka → z nią*, *jej → kobieta, panna, księżna*, *mężowi → jej*).

In [24]:
show(GPT, "VO", 8, 5, title="(żeński podmiot → z nią / w jej)")
show(GPT, "VO", 15, 4, title="(jej → kobiety, imiona żeńskie)")
show(GPT, "VO", 20, 0, title="(mąż / sama → jej, ona)")

GPT-2 (polish-gpt2-medium) – VO – warstwa 8, głowa 5 *  (żeński podmiot → z nią / w jej)
(dziewczynka, znią), (uroda, znią), (uroda, wjej), (ukazała, znią), (śliczna, znią), (przyjaciółka, znią), (rzeka, znią), (jednostka, znią), (grupka, znią), (kuzynka, znią), (grupka, jej), (księżna, znią), (przebywała, znią), (#leczkę, znią), (lekarka, znią), (zjawiła, znią), (siostra, znią), (#Gazeta, jej), (księżna, wjej), (panna, znią), (#lum, znią), (obecność, znią), (opuściła, znią), (#śmy, znią), (#wła, #pokoju), (aborcja, znią), (siostra, jej), (sama, jej), (ciążę, znią), (babka, znią), (#akim, znią), (sama, znią), (panna, wjej), (gmina, jej), (śliczna, wjej), (wtrąciła, znią), (pojawiła, znią), (brzydka, znią), (wyskoczyła, znią), (ukazała, wjej), (brzydka, jej), (uroda, jej), (pierwsza, jej), (#śmy, wjej), (dwójka, jej), (liczebność, znią), (#dora, znią), (grupa, jej), (zmarła, jej), (#śmier, znią)

GPT-2 (polish-gpt2-medium) – VO – warstwa 15, głowa 4 *  (jej → kobiety, imiona żeńskie)
(j

### C.1.3 Geografia i nazwy własne

In [25]:
show(GPT, "VO", 22, 14, title="(miasto → kraj: Oslo → Norwegia, Barcelona → Hiszpania, Bałkany)")
show(GPT, "VO", 21, 11, title="(Rosja: imiona, Putin, Petersburg)")

GPT-2 (polish-gpt2-medium) – VO – warstwa 22, głowa 14 *  (miasto → kraj: Oslo → Norwegia, Barcelona → Hiszpania, Bałkany)
(Oslo, norwe), (#ót, Vil), (#Ŝy, Portugal), (reki, Albanii), (#ille, Katalo), (#owić, Słowe), (Barcel, #Hiszpa), (#sen, Norwe), (#luzy, Katalo), (Madry, Madrycie), (#sen, norwe), (#Hiszpa, Barcelonie), (Norwegii, norwe), (Oslo, Norwe), (#iro, Portuga), (#vić, Serbii), (#Hiszpa, Madrycie), (Oslo, Norwegii), (chorwa, Chorwacji), (#Hiszpa, Barcel), (#Hiszpa, Katalo), (#erb, Etio), (#Hiszpa, Madry), (#aux, Saint), (#iro, portugal), (#vian, Chorwacji), (Tommy, Alba), (#dad, José), (#sena, Norwe), (#gody, Norwe), (chorwa, Serbii), (#des, Etio), (#Hiszpa, meksyka), (#Hiszpa, Meksyku), (hiszpańskiego, Madry), (#Hiszpa, Hiszpan), (#zo, portugal), (#vić, Bośni), (#vian, Serbii), (Barce, #Hiszpa), (#vić, Boś), (chorwa, Bośni), (hiszpańskiego, Madrycie), (#owić, Słowenii), (#rosa, Nea), (Finlandii, Bośni), (#Hiszpa, Hisz), (#hak, Stambu), (#dad, argenty), (norwe, Norwegii)

GP

### C.1.4 Pisownia
Odpowiednik głowy „British spelling”. Model uczony na polskim internecie zna **zapis bez polskich znaków** i łączy go z poprawnym (*sie → się*, *miedzy → między*, *niz → niż*) albo grupuje słowa pisane w ten sposób.

In [26]:
show(GPT, "VO", 0, 4, title="(sie → się)")
show(GPT, "VO", 19, 1, title="(miedzy → między, aniżeli → niż, osobno → oddzielnie)")
show(GPT, "VO", 19, 10, title="(słowa bez polskich znaków: wsrod, poniewaz, rowniez)")

GPT-2 (polish-gpt2-medium) – VO – warstwa 0, głowa 4 *  (sie → się)
(sie, się), (sie, coś), (sie, ją), (sie, jeszcze), (sie, też), (sie, bardzo), (sie, cały), (sie, więcej), (SIE, się), (sie, właśnie), (sie, całkiem), (sie, zupełnie), (#emon, #gly), (#nosc, się), (#scie, się), (sie, wciąż), (Camp, #iela), (sie, je), (#kacz, #alam), (sie, taki), (utrzymaniem, #rans), (#magne, #kaja), (sie, ten), (poczatku, się), (sie, na), (utrzymaniem, #iec), (sie, jakiś), (sie, za), (sie, dokładnie), (karte, #czew), (dobie, #bys), (#Roze, #clo), (#jš, byście), (sie, ciągle), (#osci, się), (sie, chwilę), (utrzymaniem, #ming), (#noscia, się), (#smy, się), (sie, jej), (utrzymaniem, #mował), (#jš, bym), (utrzymaniem, #znania), (sie, to), (#nomi, ktorym), (#śc, bym), (#lama, #gary), (#nomi, ktorzy), (ktos, się), (mozesz, się)

GPT-2 (polish-gpt2-medium) – VO – warstwa 19, głowa 1 *  (miedzy → między, aniżeli → niż, osobno → oddzielnie)
(aniżeli, niż), (niż, aniżeli), (miedzy, między), (pomiędzy, między), (

### C.1.5 Słowa pokrewne i tematy

In [27]:
show(GPT, "VO", 22, 9, title="(medycyna, śmierć: HIV → AIDS, nowotwór → onkologia)")
show(GPT, "VO", 22, 9, corpus_only=True, title="(to samo, tylko tokeny ze Składnicy)")
show(GPT, "VO", 16, 2, title="(instytucje: karetka, taksówka → kierowca, psychiatra → lekarz)")
show(GPT, "VO", 22, 12, title="(sąd, sędzia, wyrok)")
show(GPT, "VO", 10, 14, title="(synonimy: nawzajem → wzajemnie, równocześnie → jednocześnie)")

GPT-2 (polish-gpt2-medium) – VO – warstwa 22, głowa 9 *  (medycyna, śmierć: HIV → AIDS, nowotwór → onkologia)
(trucizny, truci), (nowotwor, onko), (kościach, kość), (kościach, kości), (gwałtu, gwałci), (ciało, ciała), (HIV, AIDS), (zakaź, epidemio), (grypy, szczepion), (wirusem, wirus), (wirusem, wirusa), (samobój, samobójstwa), (trup, trupa), (pożarze, pożaru), (szczepionki, szczepion), (zakaźnych, epidemio), (grypy, szczepionki), (ciał, ciała), (grypy, wirus), (nowotwor, nowotw), (ofierze, ofiara), (raka, onko), (zwłok, ciał), (trupa, trup), (samobój, samobójstwo), (epidem, epidemio), (zgwał, gwałci), (czaszki, czaszkę), (grypy, epidemio), (#onoś, Śmierci), (kości, kość), (zwłok, ciała), (AIDS, HIV), (pożarze, pożar), (ciała, ciało), (ciało, ciał), (ciałem, ciała), (zakaźnych, zakaź), (kość, kości), (szczepionki, szczepionka), (ciała, ciał), (zakaź, zakaźnych), (języki, języ), (zakaźnych, wirusa), (morderstwo, morderca), (wirusem, zaraza), (języku, języ), (gwałtu, gwałtów), (energii,

GPT-2 (polish-gpt2-medium) – VO – warstwa 22, głowa 9 * **  (to samo, tylko tokeny ze Składnicy)
(kościach, kości), (ciało, ciała), (HIV, AIDS), (wirusem, wirus), (wirusem, wirusa), (trup, trupa), (grypy, szczepionki), (ciał, ciała), (grypy, wirus), (zwłok, ciał), (trupa, trup), (samobój, samobójstwo), (zwłok, ciała), (AIDS, HIV), (ciała, ciało), (ciało, ciał), (ciałem, ciała), (ciała, ciał), (języki, języ), (wirusem, zaraza), (języku, języ), (astronomi, astro), (grypy, wiru), (poległych, grobów), (język, języ), (grypy, wirusa), (nowotwor, nowotworów), (wirusem, zaraż), (chorobami, wirus), (grypy, immuno), (energetyczne, energetyczna), (chorób, choroby), (języka, języ), (ciało, ciałem), (ciałem, ciał), (atomowej, atom), (chorobami, wirusa), (złamanie, złamania), (zwłok, ciało), (ciałem, ciało), (HIV, wirusem), (chorób, chorobami), (zatru, truci), (śmiercią, Śmierci), (kost, kości), (pożaru, pożar), (energetycznego, energetyczna), (śmiercią, śmierć), (wirusem, wiru), (zwłok, ciałem)

GP

### C.1.6 Formy adresatywne (polska specyfika)
*pan / pani* (forma grzecznościowa) → *twój, ciebie, myślisz*: głowa łączy grzecznościowe i bezpośrednie zwroty do rozmówcy.

In [28]:
show(GPT, "VO", 0, 13, title="(pan → twój, ciebie, sądzisz)")
show(GPT, "VO", 9, 6, variant="raw", title="(dla porównania: lista bez wykluczania par (x, x); głowa w dużej części kopiuje token)")

GPT-2 (polish-gpt2-medium) – VO – warstwa 0, głowa 13 *  (pan → twój, ciebie, sądzisz)
(pan, twój), (pan, twoje), (pan, twoim), (pan, Twoim), (pan, Twój), (pan, twoją), (pan, twoimi), (pan, Twoje), (pan, twojej), (pan, twoja), (pan, Twoją), (pan, twoi), (pan, twoich), (pan, ciebie), (pan, tobie), (pan, twej), (pan, #Twoje), (pan, sądzisz), (pani, twoim), (pan, Twoich), (pan, Twojej), (pan, Twojego), (pani, twój), (pani, twoje), (pan, twojego), (pan, Twoja), (pan, #niłeś), (pan, myślisz), (pan, Was), (pani, Twoim), (pan, Tobie), (pan, wasz), (pan, musiałeś), (pan, #łeś), (pana, twój), (panu, twój), (pan, tobą), (pan, miałeś), (pani, ciebie), (pan, uważasz), (pan, waszym), (pan, #Twój), (pana, twoje), (pan, sądzicie), (pani, twoją), (pani, twoimi), (pani, Twoje), (panu, twoje), (pan, Ciebie), (pan, słyszysz)

GPT-2 (polish-gpt2-medium) – VO – warstwa 9, głowa 6   (dla porównania: lista bez wykluczania par (x, x); głowa w dużej części kopiuje token)
(#ceu, #ceu), (#aryka, #aryka), (zaludn

### C.1: HerBERT
Dla BERT-a interpretowalnych głów jest znacznie mniej (patrz dyskusja). Przykłady najlepszych:

In [29]:
show(BERT, "VO", 4, 4, title="(odmiana: wyglądają → wygląda, propozycji → propozycje)")
show(BERT, "VO", 11, 8, title="(odmiana + pokrewne: funkcji → funkcje, gmail → hotmail, Resort → Departament)")
show(BERT, "VO", 6, 0, title="(przysłówki: ciągle → wciąż, zapewne → pewnie, ale też szum)")

BERT (HerBERT-base) – VO – warstwa 4, głowa 4 *  (odmiana: wyglądają → wygląda, propozycji → propozycje)
(wyglądają, wygląda), (propozycji, propozycje), (wyglądają, wyglądał), (wysokość, wysokości), (propozycje, propozycji), (okoliczności-, opowieści), (pozostają, fotel), (wyglądają, motyw-), (źródeł, źródła), (krop-, linia), (pokój, pokoju), (pozostają, pozostał), (wyglądają, przebieg-), (systemów, pakiet), (faktycznie, rzeczywiście), (czuję, motyw-), (związek, związku), (pozostają, pozostał-), (pozostają, wyraz), (linii, linia), (wypełni-, treści), (pokoju, akta), (rusz-, pakiet), (istnieją, rzeczywiście), (faktycznie, motyw-), (wyraz, treści), (wypełni-, wówczas), (różnic-, metafor-), (rośnie, rosną), (niejszym, dobra), (pozostają, wyposażenie), (wyglądają, wówczas), (krop-, faktycznie), (rusz-, brzmi), (uczucie, motyw-), (wypełni-, zabezpiecz-), (Rodzin, dokumenty), (istnieją, faktycznie), (materiałów, materiał), (kompon-, układ-), (wyglądają, zabezpiecz-), (wyglądają, motyw), (wia

## C.2: macierze $W_{QK}$

Pary *(token-zapytanie → token-klucz)*: na co patrzy dany token, jeśli głowa nie ma innych wskazówek.

### Zgoda gramatyczna

In [30]:
show(GPT, "QK", 8, 15, title="(zgoda przecząca: ani → żaden, żadną, niczyjej)")
show(GPT, "QK", 16, 12, title="(która / żeńskie czasowniki → jej, ją)")
show(GPT, "QK", 18, 0, title="(zaimki → zaimki tego samego rodzaju: ją → jej, nich → ich)")
show(GPT, "QK", 11, 12, title="(tryb przypuszczający: gdybym → musiałbym, miałbym)")
show(GPT, "QK", 23, 15, title="(pan ↔ pański)")

GPT-2 (polish-gpt2-medium) – QK – warstwa 8, głowa 15 *  (zgoda przecząca: ani → żaden, żadną, niczyjej)
(ani, żadnymi), (ani, żadną), (ani, żadnej), (ani, żad), (ani, zadnej), (żadnymi, żadną), (ani, niczyjej), (ani, żadne), (ani, żadnych), (ani, niczy), (żadnymi, żadnej), (żadnych, żadnymi), (ani, żadnemu), (ani, żadnym), (żadnego, żadnymi), (ani, żadnego), (żadnymi, żad), (żadnego, żadnej), (#ette, Bast), (żadnymi, żadnego), (zadnego, żadnymi), (żadnym, żadną), (żadnymi, zadnej), (żadnego, żadną), (zadnego, żadnej), (ani, zadnego), (żadną, żadnej), (żadnej, żadnymi), (żadnymi, żadne), (#dell, #smo), (zadnej, żadnej), (zadnych, żadnymi), (#ody, #the), (#cum, #the), (żadnymi, żadnym), (żadnym, żadnymi), (żadnymi, żadnych), (żadnym, żadnej), (żadnych, żadnej), (żadnych, żadną), (jakbym, żebym), (#rzej, #zon), (żadnemu, żadnymi), (ani, zadnych), (żeś, żebyś), (żadnemu, żadnej), (żadnego, zadnej), (żadnemu, żadną), (zadnego, zadnej), (żadnemu, żad)

GPT-2 (polish-gpt2-medium) – QK – wars

### Tematy, geografia, liczebniki, rejestr

In [31]:
show(GPT, "QK", 22, 14, title="(Hiszpania ↔ Portugalia, Francja, Włochy)")
show(GPT, "QK", 21, 3, title="(pandemia, strajki)")
show(GPT, "QK", 22, 9, title="(pogrzeb, grób → zmarłych, ekshumacja)")
show(GPT, "QK", 21, 12, title="(sport: piłkarz → zawodnicy, mecz, turniej)")
show(GPT, "QK", 20, 2, title="(lewa ↔ prawa, liczebniki porządkowe, godziny)")
show(GPT, "QK", 20, 12, title="(rejestr internetowy: xD, https, wulgaryzmy)")

GPT-2 (polish-gpt2-medium) – QK – warstwa 22, głowa 14 *  (Hiszpania ↔ Portugalia, Francja, Włochy)
(Hiszpanii, Portugal), (Hiszpania, Hiszpa), (Hiszpanii, Hiszpa), (Hiszpa, Portugal), (Hiszpania, Portugal), (hiszpa, Portugal), (Hiszpan, Hiszpa), (Hiszpan, Portugal), (Francji, Portugal), (Hiszpanii, Hiszpan), (Hiszpa, Hiszpan), (Portugalii, Hiszpa), (Hiszpania, Hiszpan), (hiszpańskiego, Portugal), (Francji, Hiszpa), (#Hiszpa, Portugal), (Hiszpa, portugal), (Francja, Hiszpa), (Chile, Hiszpa), (Portugal, Hiszpa), (portugal, Hiszpa), (Francję, Hiszpa), (Hiszpa, Hiszpanii), (Hiszpanii, portugal), (hiszpańskiej, Portugal), (Francuzów, Hiszpa), (Włoszech, Hiszpa), (Hiszpanii, włoskim), (Francji, Hiszpanii), (Hiszpan, włoskim), (Hiszpania, Hiszpanii), (hiszpańskiej, Hiszpa), (Hiszpania, portugal), (hiszpański, Portugal), (Portugalii, Portugal), (Francją, Hiszpa), (Włosi, Hiszpa), (argenty, Portugal), (Francja, Portugal), (Portuga, Hiszpa), (Francu, Portugal), (francuską, Hiszpa), (Włoszech, P

### C.2: HerBERT

In [32]:
show(BERT, "QK", 2, 9, title="(odmiana: wyników → wyniki, całości → całość)")
show(BERT, "QK", 1, 1, title="(typowa głowa: token-hub „zazwyczaj” → przypadkowe podsłowa)")

BERT (HerBERT-base) – QK – warstwa 2, głowa 9 *  (odmiana: wyników → wyniki, całości → całość)
(wyników, wyniki), (całości, całość), (wyniki, wyników), (trwał-, Two-), (dwie, drugą), (całość, całości), (trwał-, Chro-), (wynik, wyniki), (własne, własnych), (wyników, wynik), (wyniku, wynik), (wyniki, wynik), (wyniku, wyniki), (godzinę, godziny), (wykonania, wykon-), (ofert, ofertę), (dotychczas-, dotychczasowa), (wynik-, wyniki), (godziny, godzinę), (wynik, wyników), (dwie, dwóch), (własną, własnych), (wyglądają, wygląda), (wynik, wyniku), (Two-, trwał-), (poprawia, wysoką), (byłoby, gdyby), (dotychczas-, dotychczasowych), (wyglądał, wygląda), (dwie, drugiej), (Rekord, rekor-), (wykonania, cios-), (ofertę, ofert), (OWE-, ON-), (Biego-, bieg-), (drugą, dwie), (dotychczas-, zastąpił), (własne, własną), (wykonania, trwał), (miałem, miałam), (najwyższej, wysoką), (wyników, wynik-), (wyniki, wynik-), (wykonania, komor-), (całość, całością), (poszczególnych, niektórych), (jakich, Jakie), (wyśc

## Podsumowanie części 2

**GPT-2 (polish-gpt2-medium): projekcja działa i daje czytelne listy, coraz lepsze z głębokością.**
- Interpretowalność rośnie z warstwą. Odsetek par morfologicznych w top-50 $W_{VO}$ wynosi 0,015 w dolnej, 0,079 w środkowej i 0,154 w górnej tercji warstw ($W_{QK}$: 0,004 / 0,035 / 0,121). W warstwach 0–5 listy to głównie fragmenty podsłów (`#…`) bez wyraźnej reguły.
- 46 z 384 głów $W_{VO}$ ma co najmniej połowę top-50 par typu $(x, x)$, czyli **kopiuje** token, na który patrzy. To samo zjawisko opisują autorzy.
- Kategorie z artykułu mają polskie odpowiedniki, ale ukształtowane przez polską gramatykę:
  - „dokańczanie słów” to u nas przede wszystkim **odmiana** (*rocznica → rocznicy*, *#nicą → #nicami*, końcówki *-byśmy/-byście*). Polszczyzna ma bogatą fleksję, więc najczęstszą relacją między tokenami jest „inna forma tego samego wyrazu”;
  - **rodzaj** jest gramatyczny, nie tylko biologiczny: żeńskie rzeczowniki, imiesłowy i czasowniki dopisują żeńskie zaimki (*dziewczynka → z nią*, *mężowi → jej*). To szersze niż *his → her* u autorów;
  - **geografia** wygląda jak u autorów (*Oslo → Norwegia*, *Barcelona → Hiszpania*, głowa „rosyjska”: *Dmitrij → Władimir → Putin*);
  - odpowiednikiem „pisowni brytyjskiej” jest **zapis bez polskich znaków** (*sie → się*, *miedzy → między*, *wsrod, poniewaz, rowniez*);
  - **tematy**: medycyna (*HIV → AIDS*), sądownictwo, transport i służby, synonimy;
  - polską specyfiką są **formy adresatywne** (*pan → twój, ciebie*).
- $W_{QK}$ pokazuje głowy **zgody składniowej**, które w angielskim prawie nie mają odpowiednika: **zgoda przecząca** (*ani → żaden, żadną*: polskie przeczenie wymaga zaimków przeczących), zgoda rodzaju zaimków (*która → jej*), tryb przypuszczający (*gdybym → musiałbym*), *pan ↔ pański*. Do tego głowy tematyczne (pandemia, pogrzeb, sport) i głowa rejestru internetowego (*xD, https*, wulgaryzmy).
- Listy zdradzają **dane treningowe**: tokeny z Wykopu (*AnonimoweMirkoWyznania*, *#zwykopem*, *Zaakceptował*) pojawiają się jako „śmieciowe” tokeny w wielu głowach, podobnie jak w artykule pojawiały się tokeny z angielskiego internetu.

**HerBERT: projekcja działa dużo słabiej.**
- Średni udział par morfologicznych w top-50 to 0,013 ($W_{VO}$) i 0,006 ($W_{QK}$) wobec 0,083 i 0,053 w GPT-2. Tylko 1 głowa $W_{VO}$ ma ≥ 30% par morfologicznych (GPT-2: 40).
- Wiele list zdominowały **tokeny-huby** (*zazwyczaj, wówczas, zapewne, Również*): jeden token jest „zapytaniem” w prawie wszystkich parach. Sprawdziliśmy ich geometrię: *zazwyczaj, wówczas, zapewne, Również* należą do ok. 1–3% tokenów o **najbardziej skrajnej (ujemnej) składowej wzdłuż średniego wektora $E$** i mają ponadprzeciętną normę (top 3–9%). Po przemnożeniu przez macierze głowy dają więc duże wartości niezależnie od drugiego tokenu i wygrywają top-k w wielu głowach.
- Wynika to z założeń metody, o których piszą autorzy:
  - projekcja przez $E^T$ zakłada liniową głowę LM, jak w GPT-2. BERT ma nieliniową głowę MLM (dense + GELU + LayerNorm);
  - BERT ma też post-LayerNorm po każdym podmodule, więc pominięcie LN jest w nim większym uproszczeniem niż w GPT-2 (pre-LN);
  - mimo to kilka głów (np. 4.4, 11.8 w $W_{VO}$, 2.9 w $W_{QK}$) pokazuje tę samą morfologię co GPT-2.
- Naturalną poprawką (niesprawdzoną tutaj) byłoby wycentrowanie $E$ przed rzutowaniem, czyli usunięcie średniego wektora osadzeń, wzdłuż którego leżą huby.

**Ograniczenia.** Wybór głów jest ręczny, tak jak w artykule (to ilustracja, nie miara). Dlatego obok przykładów podajemy statystyki wszystkich głów. Filtr tokenów (tylko litery, ≥ 2 znaki) usuwa liczby i interpunkcję, więc głowy zajmujące się nimi są niewidoczne.